In [1]:
import pandas as pd
import numpy as np

In [2]:
met = pd.read_csv("metab_concat.csv")
met.rename(columns={'CHEM_ID':'ProbeID',"CHEMICAL_NAME":"Name"}, inplace=True)
metm = pd.read_csv('all_divmedlog_omit-survival.txt',sep="\t")

In [ ]:
#log the protien. change 0's to na in protien. normalize protien. convert to long. remove all na's 
#is the metab in log?

In [3]:
exp = pd.read_csv('panda-maier-mouse-exercise-tpm-2023-expr.txt',sep="\t")
expm = pd.read_csv('panda-maier-mouse-exercise-tpm-2023-survival.txt',sep="\t")
print(metm['c tissue'].unique())

#fix erronius M's to F's in BF original names
rm2f = expm.loc[(expm['c Tissue']=="BF") & (expm['c Sex']=="F") & (expm['c sample.names.orignal'].str.contains("M")),'c sample.names.orignal'].str.replace('M',"F").to_list()
expm.loc[(expm['c Tissue']=="BF") & (expm['c Sex']=="F") & (expm['c sample.names.orignal'].str.contains("M")),'c sample.names.orignal'] = rm2f

f2sd = {'BF':'BF', 'Quad':'QUAD', 'Gas':'GAS', 'Heart':'HEA', 'Liver':'LIV', 'Lung':'LUN'}
expm["c Tissue"] = expm["c Tissue"].replace(f2sd) #convert tissue names to the metab tissue names
expm = expm[expm['c Tissue'].isin(metm['c tissue'].unique())] # remove extra tissues
expm['mnum'] = [i.split('_')[1] for i in expm['c sample.names.orignal']]
expm['newname'] = expm['mnum']+"_"+expm['c Group']+"_"+expm['c ZT'].astype(str)+"_"+expm['c Tissue'] #NEED TO FIGURE OUT THE BF SEX DISCREPANCY
expm['newname'] = expm['newname'].str.upper()

#CUSTOM FIXES TO INDIVIDUALS
#M34 to M36
expm.loc[expm['mnum'] == "M34",['mnum','newname']] = ['M36','M36_RW_16_QUAD']


exp = exp.loc[:,exp.columns.isin(list(expm['ArrayId'])+['ProbeID','Name'])]
nnd = {expm['ArrayId'][i]:expm['newname'][i] for i in expm.index}
exp.rename(columns=nnd,inplace=True) #name match the metab samples

['BF' 'QUAD' 'GAS' 'HEA' 'LIV' 'LUN']


In [4]:
#prot = pd.read_excel('proteomics/251217_ProfPanda_muscle_total.xlsx',sheet_name=[0,1]) #2 is soleus
prot = pd.read_excel('proteomics/260120_ProfPanda_muscle_total.xlsx',sheet_name=[0,1,2]) #1 quad #2 gas #3 bf
protm = pd.read_excel('proteomics/Japan Proteomics_Gery.xlsx',sheet_name=1)
prot[0].columns = prot[0].columns.str.removesuffix('.raw') #quad
prot[1].columns = prot[1].columns.str.removesuffix('.raw') #gas
pmd = {str(protm['Japan IDs'][i]):protm['Gery ID\'s'][i] for i in protm.index}
prot[0].rename(columns=pmd, inplace=True)
prot[1].rename(columns=pmd, inplace=True)

quad_num2_id = {i.split("_")[0]:i for i in metm['ArrayID'] if "QUAD" in i}
gas_num2_id = {i.split("_")[0]:i for i in metm['ArrayID'] if "GAS" in i}

prot[0].rename(columns=quad_num2_id, inplace=True)
prot[1].rename(columns=gas_num2_id, inplace=True)


prot[2].columns = prot[2].columns.str.removesuffix('.raw') #gas
prot[2].rename(columns=pmd, inplace=True)
bf_num2_id = {i.split("_")[0]:i for i in metm['ArrayID'] if "BF" in i}
prot[2].rename(columns=bf_num2_id, inplace=True)

In [5]:
#fix protien names
prot[0]['Protein.Names'] = prot[0]['Protein.Names'].str.removesuffix("_MOUSE")
nn = []
for i in prot[0]['Protein.Names']:
    if ';' in i:
        cm = i.split(';')
        cm = list(pd.Series(cm).str.removesuffix("_MOUSE"))
        cm = ":".join(cm)
        #print(cm)
        nn+=[cm]
    else:
        nn += [i]
prot[0]['Protein.Names'] = nn


prot[1]['Protein.Names'] = prot[1]['Protein.Names'].str.removesuffix("_MOUSE")
nn = []
for i in prot[1]['Protein.Names']:
    if ';' in i:
        cm = i.split(';')
        cm = list(pd.Series(cm).str.removesuffix("_MOUSE"))
        cm = ":".join(cm)
        #print(cm)
        nn+=[cm]
    else:
        nn += [i]
prot[1]['Protein.Names'] = nn


prot[2]['Protein.Names'] = prot[2]['Protein.Names'].str.removesuffix("_MOUSE")
nn = []
for i in prot[2]['Protein.Names']:
    if ';' in i:
        cm = i.split(';')
        cm = list(pd.Series(cm).str.removesuffix("_MOUSE"))
        cm = ":".join(cm)
        #print(cm)
        nn+=[cm]
    else:
        nn += [i]
prot[2]['Protein.Names'] = nn

In [6]:
#convert 0's to na's
prot[0].replace(0, np.nan, inplace=True)
prot[1].replace(0, np.nan, inplace=True)
prot[2].replace(0, np.nan, inplace=True)

In [7]:
#median has skipna default to true
sampcols = prot[0].columns[4:-2]
p0 = prot[0][sampcols]
p0 = np.log2(p0) #log first, then subtract
p0 = p0.sub(p0.median())
p0.insert(0,'ProbeID',prot[0]['Protein.Names'])

sampcols = prot[1].columns[4:-1] #THIS ONE GAS DOES NOT HAVE Av. Filtering column at the end like 0(QUAD) DOES
p1 = prot[1][sampcols]
p1 = np.log2(p1) #log first, then subtract
p1 = p1.sub(p1.median())
p1.insert(0,'ProbeID',prot[1]['Protein.Names'])

sampcols = prot[2].columns[4:-2]#bf has both like quad
p2 = prot[2][sampcols]
p2 = np.log2(p2) #log first, then subtract
p2 = p2.sub(p2.median())
p2.insert(0,'ProbeID',prot[2]['Protein.Names'])

In [8]:
print(len(p0.columns),len(p1.columns),len(p2.columns),len(met.columns),len(exp.columns))

25 25 25 194 146


In [9]:
import pandas as pd
from typing import List, Iterable

def combine_to_long(
    dfs: List[pd.DataFrame],
    view_suffixes: Iterable[str] = ("rna", "metabolite", "protein", "protein", "protein"),
    probe_col: str = "ProbeID",
    exclude_cols: Iterable[str] = ("Name",),
) -> pd.DataFrame:
    """
    Combine wide dataframes into one long-form dataframe with columns:
    sample, group, feature, view, value.

    The length of `view_suffixes` must match the number of dataframes.
    """

    if len(dfs) != len(view_suffixes):
        raise ValueError(
            f"Number of dataframes ({len(dfs)}) must match "
            f"number of view_suffixes ({len(view_suffixes)})"
        )

    exclude_cols = set(exclude_cols) | {probe_col}
    out_parts = []

    for i, (df, view_sfx) in enumerate(zip(dfs, view_suffixes), start=1):
        if probe_col not in df.columns:
            raise KeyError(f"DataFrame {i} missing required column '{probe_col}'")

        # value columns = everything except ProbeID and excluded columns
        value_cols = [c for c in df.columns if c not in exclude_cols]
        if not value_cols:
            raise ValueError(
                f"DataFrame {i} has no value columns after excluding {exclude_cols}"
            )

        long = df.melt(
            id_vars=[probe_col],
            value_vars=value_cols,
            var_name="_col",
            value_name="value",
        ).rename(columns={probe_col: "feature"})

        # split column name into sample and group using LAST underscore
        sample_group = long["_col"].astype(str).str.rsplit("_", n=1, expand=True)
        if sample_group.shape[1] != 2:
            raise ValueError(
                f"DataFrame {i}: expected column names like 'sample_group'"
            )

        long["sample"] = sample_group[0]
        long["group"] = sample_group[1]

        # construct view
        long["view"] = long["group"].astype(str) + "_" + str(view_sfx)

        out_parts.append(long[["sample", "group", "feature", "view", "value"]])

    return pd.concat(out_parts, ignore_index=True)

outwna = combine_to_long(
    dfs=[exp, met, p0, p1, p2],
    #view_suffixes=("rna", "metabolite", "protein", "protein", "protein"),
)

outwna["feature"] = outwna["feature"].astype(str)



In [ ]:
outwna.to_csv('long_exp_mofa_input_wna_lp.tsv',sep="\t",index=None)

In [10]:
outwnafilter = outwna.groupby(["feature", "view"]).filter(
    lambda g: not g["value"].isna().all()
)
outwna.to_csv('long_exp_mofa_input_wnaf_lp.tsv',sep="\t",index=None)

In [11]:
###DROP NAS ###
print(len(outwna))
outnona = outwna.dropna()
len(outnona)
outnona.to_csv('long_exp_mofa_input_nona_lp.tsv',sep="\t",index=None)

4372248


In [15]:
outwna.sort_values(["feature", "view"])

,sample,group,feature,view,value
3723918,M31_RW_4,BF,100000007,BF_metabolite,2.097766
3724956,M32_RW_4,BF,100000007,BF_metabolite,2.190322
3725994,M33_RW_4,BF,100000007,BF_metabolite,2.157353
3727032,M34_RW_4,BF,100000007,BF_metabolite,2.047279
3728070,M36_RW_16,BF,100000007,BF_metabolite,2.182240
...,...,...,...,...,...
4056857,F32_RW_4,QUAD,ZZEF1,QUAD_protein,-5.376391
4063601,F33_RW_4,QUAD,ZZEF1,QUAD_protein,NaN
4070345,F36_RW_16,QUAD,ZZEF1,QUAD_protein,-5.908022
4077089,F37_RW_16,QUAD,ZZEF1,QUAD_protein,NaN


In [12]:
print(len(outwna))
print(len(outwnafilter))
print(len(outnona))

4372248
4301304
4210475


In [ ]:
out.loc[(out['view']=="BF_rna"),'value'].plot(kind="hist",bins=100)

In [ ]:
out.loc[(out['view']=="BF_protein"),'value'].plot(kind="hist",bins=100)

In [ ]:
out.loc[(out['view']=="BF_metabolite"),'value'].plot(kind="hist",bins=100)

In [ ]:
met